In [7]:
import numpy as np

# 三维杆单元刚度矩阵计算
def truss3d_element_stiffness(x1, x2, E, A):
    x1 = np.array(x1, dtype=float)
    x2 = np.array(x2, dtype=float)
    dx = x2[0] - x1[0]
    dy = x2[1] - x1[1]
    dz = x2[2] - x1[2]

    L = np.sqrt(dx**2 + dy**2 + dz**2)
    if L < 1e-12:
        raise ValueError("错误：节点重合，单元退化！")

    cx = dx / L
    cy = dy / L
    cz = dz / L

    C = np.array([
        [cx*cx, cx*cy, cx*cz],
        [cx*cy, cy*cy, cy*cz],
        [cx*cz, cy*cz, cz*cz]
    ])

    Ke = (E * A / L) * np.block([[C, -C], [-C, C]])
    return L, np.array([cx, cy, cz]), Ke

# 应变、应力、轴力
def truss3d_element_stress(x1, x2, E, A, de):
    x1 = np.array(x1, dtype=float)
    x2 = np.array(x2, dtype=float)
    de = np.array(de, dtype=float).reshape(6, 1)

    dx = x2[0] - x1[0]
    dy = x2[1] - x1[1]
    dz = x2[2] - x1[2]
    L = np.sqrt(dx**2 + dy**2 + dz**2)

    cx = dx / L
    cy = dy / L
    cz = dz / L

    B = np.array([[-cx, -cy, -cz, cx, cy, cz]]) / L
    eps = float(B @ de)
    sigma = E * eps
    N = sigma * A
    return eps, sigma, N


# 主程序：

if __name__ == "__main__":

#算例 1 
    print("=" * 60)
    print("算例1：沿x轴的一维杆单元")
    print("=" * 60)
    x1_1 = [0, 0, 0]
    x2_1 = [2, 0, 0]
    E1 = 200e9
    A1 = 1.0e-4
    de1 = [0, 0, 0, 1.0e-3, 0, 0]

    L1, dir1, Ke1 = truss3d_element_stiffness(x1_1, x2_1, E1, A1)
    eps1, sig1, N1 = truss3d_element_stress(x1_1, x2_1, E1, A1, de1)

    print(f"单元长度 L = {L1:.4f} m")
    print(f"方向余弦 (cx, cy, cz) = {np.round(dir1, 4)}")
    print("\n单元刚度矩阵 Ke (N/m):")
    print(np.round(Ke1, 4))
    print(f"\n轴向应变 ε = {eps1:.6e}")
    print(f"轴向应力 σ = {sig1/1e6:.2f} MPa")
    print(f"轴向轴力 N = {N1:.2f} N")

# 算例 2 
    print("\n" + "=" * 60)
    print("算例2：空间任意方向杆单元")
    print("=" * 60)
    x1_2 = [0, 0, 0]
    x2_2 = [1, 2, 2]
    E2 = 210e9
    A2 = 2.0e-4
    de2 = [0, 0, 0, 1.0e-3, 2.0e-3, 2.0e-3]

    L2, dir2, Ke2 = truss3d_element_stiffness(x1_2, x2_2, E2, A2)
    eps2, sig2, N2 = truss3d_element_stress(x1_2, x2_2, E2, A2, de2)

    print(f"单元长度 L = {L2:.4f} m")
    print(f"方向余弦 (cx, cy, cz) = {np.round(dir2, 4)}")
    print("\n单元刚度矩阵 Ke (N/m):")
    print(np.round(Ke2, 4))
    print(f"\n轴向应变 ε = {eps2:.6e}")
    print(f"轴向应力 σ = {sig2/1e6:.2f} MPa")
    print(f"轴向轴力 N = {N2:.2f} N")

# 矩阵性质验证 
    print("\n" + "=" * 60)
    print("单元刚度矩阵性质验证")
    print("=" * 60)

# 1. 对称性
    sym_check = np.allclose(Ke2, Ke2.T)
    print(f"1. 对称性验证：{sym_check}")

# 2. 奇异性（行列式）
    det_Ke = np.linalg.det(Ke2)
    print(f"2. 行列式值：{det_Ke:.4e}（接近0，说明矩阵奇异）")

# 3. 特征值
    eig_vals = np.linalg.eigvals(Ke2).real
    eig_sorted = np.sort(eig_vals)
    min_eig = np.min(eig_vals)
    all_non_neg = np.all(eig_vals >= -1e-6)
    print(f"3. 特征值（从小到大）：{np.round(eig_sorted, 4)}")
    print(f"   最小特征值：{min_eig:.4e}（接近0，对应刚体位移）")
    print(f"   所有特征值非负：{all_non_neg}")

# 4. 刚体位移
    de_rigid = [1, 1, 1, 1, 1, 1]
    Fe_rigid = Ke2 @ np.array(de_rigid)
    max_force = np.max(np.abs(Fe_rigid))
    print("\n4. 刚体位移验证（整体平移）：")
    print(f"   节点内力列阵 Fe = {np.round(Fe_rigid, 4)}（接近零向量）")
    print(f"   内力最大值：{max_force:.4e} N")

#  物理意义验证 
    print("\n" + "=" * 60)
    print("刚度矩阵物理意义验证")
    print("=" * 60)

    j = 4   # 第5自由度
    de_j = np.zeros(6)
    de_j[j] = 1.0
    Fe = Ke2 @ de_j

    print(f"令第2个自由度（节点2的y方向）位移为1，其他固定：")
    print(f"计算得到的节点内力 Fe = {np.round(Fe, 4)}")
    print(f"刚度矩阵第5列 Ke[:,{j}] = {np.round(Ke2[:, j], 4)}")
    print(f"两者相等：{np.allclose(Fe, Ke2[:, j])}")

    

算例1：沿x轴的一维杆单元
单元长度 L = 2.0000 m
方向余弦 (cx, cy, cz) = [1. 0. 0.]

单元刚度矩阵 Ke (N/m):
[[ 10000000.         0.         0. -10000000.        -0.        -0.]
 [        0.         0.         0.        -0.        -0.        -0.]
 [        0.         0.         0.        -0.        -0.        -0.]
 [-10000000.        -0.        -0.  10000000.         0.         0.]
 [       -0.        -0.        -0.         0.         0.         0.]
 [       -0.        -0.        -0.         0.         0.         0.]]

轴向应变 ε = 5.000000e-04
轴向应力 σ = 100.00 MPa
轴向轴力 N = 10000.00 N

算例2：空间任意方向杆单元
单元长度 L = 3.0000 m
方向余弦 (cx, cy, cz) = [0.3333 0.6667 0.6667]

单元刚度矩阵 Ke (N/m):
[[ 1555555.5556  3111111.1111  3111111.1111 -1555555.5556 -3111111.1111
  -3111111.1111]
 [ 3111111.1111  6222222.2222  6222222.2222 -3111111.1111 -6222222.2222
  -6222222.2222]
 [ 3111111.1111  6222222.2222  6222222.2222 -3111111.1111 -6222222.2222
  -6222222.2222]
 [-1555555.5556 -3111111.1111 -3111111.1111  1555555.5556  3111111.1111
   31111

C:\Users\23497\AppData\Local\Temp\ipykernel_12900\4054526532.py:44: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  eps = float(B @ de)
